# StageBridge
**Cross-modal spatial–snRNA-seq bridge for lung precursor-to-LUAD progression**

---

This notebook is the **entrypoint/orchestrator** for StageBridge data workflows:

1. Build interim AnnData artifacts from GEO raw files (`snrna_*`, `spatial_*`).
2. Run HLCA full-scale mapping for snRNA (`snrna_hlca_latent_full.h5ad` + labels parquet).
3. Run Tangram projection of HLCA-labeled snRNA onto spatial spots.
4. Continue with downstream preprocessing, training, and benchmark/eval cells.

By default, heavy build/mapping steps are disabled behind toggles in section **0A**.
Enable only what you want to run in this session.


## 0 — Imports & Configuration

In [1]:
import json
import shlex
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # change to 'inline' for interactive display
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
from tqdm.auto import tqdm

# ── GPU check ─────────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU : {GPU_NAME}  ({GPU_MEM_GB:.1f} GB VRAM)")
        print(f"CUDA: {torch.version.cuda}")
        USE_GPU = True
    else:
        print("PyTorch installed but no CUDA GPU found — running on CPU.")
        USE_GPU = False
except ImportError:
    print("PyTorch not installed.")
    USE_GPU = False

# ── Optional packages ─────────────────────────────────────────────────────
try:
    import squidpy as sq
    SQUIDPY = True
    print(f"squidpy {sq.__version__}")
except ImportError:
    SQUIDPY = False
    print("squidpy not installed — spatial stats steps will be skipped")

try:
    import harmonypy
    HARMONY = True
    print("harmonypy available")
except ImportError:
    HARMONY = False
    print("harmonypy not installed — batch correction step will be skipped")

try:
    import scvi
    SCVI = True
    print(f"scvi-tools {scvi.__version__}")
    if USE_GPU:
        scvi.settings.dl_num_workers = 4   # data loader workers
        # scvi-tools auto-detects GPU; confirm with:
        print(f"  scvi accelerator: {'gpu' if USE_GPU else 'cpu'}")
except ImportError:
    SCVI = False
    print("scvi-tools not installed — scVI/scANVI steps will be skipped")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1

# ── Repo root on path (if not installed as a package) ─────────────────────
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from stagebridge.logging_utils import configure_root_logger
configure_root_logger()

from stagebridge import config
from stagebridge.preprocessing.harmonize import (
    intersect_genes,
    normalize_log1p,
    select_hvg,
    pca_fit_transform_snrna,
    pca_transform_spatial,
    run_harmony,
    run_umap,
)

FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRepo root : {REPO_ROOT}")
print(f"Figures   : {FIGURES_DIR}")


GPU : NVIDIA RTX 4000 Ada Generation  (21.5 GB VRAM)
CUDA: 12.8


squidpy 1.8.1
harmonypy available


scvi-tools 1.4.2
  scvi accelerator: gpu

Repo root : /home/ajbook/projects/StageBridge
Figures   : /home/ajbook/projects/StageBridge/outputs/figures


## 0A — Pipeline Controls (AnnData + HLCA + Tangram)

In [2]:
# ── Build/mapping toggles (set True to execute that step) ────────────────
PIPELINE_DATA = "local"          # Hydra data config: local | default
PIPELINE_EXPERIMENT = "smoke"    # smoke | full for AnnData build scripts
PIPELINE_OVERRIDES = []          # e.g. ["pipeline.max_workers=8"]

RUN_BUILD_SNRNA = False
RUN_BUILD_SPATIAL = False

# HLCA mapping is typically run on the full snRNA dataset.
RUN_HLCA_MAPPING = False
HLCA_EXPERIMENT = "full"
HLCA_OVERRIDES = []              # e.g. ["hlca.surgery_epochs=200"]
RUN_HLCA_EVAL = False
HLCA_EVAL_OVERRIDES = []         # e.g. [+hlca_eval.max_query_cells_knn=200000]

# Tangram projection from HLCA-labeled snRNA -> spatial spots.
RUN_TANGRAM_MAPPING = True
TANGRAM_EXPERIMENT = "full"
TANGRAM_OVERRIDES = [
    "tangram.show_progress=true",
    "tangram.device=cpu",
    "tangram.max_training_genes=1000",
    "tangram.num_epochs=200",
]           # e.g. ["tangram.num_epochs=300"]

# What this notebook should load after optional build steps.
LOAD_EXPERIMENT = PIPELINE_EXPERIMENT   # smoke | full
ATTACH_PRECOMPUTED_HLCA = True

print("Controls configured.")
print(f"  data config         : {PIPELINE_DATA}")
print(f"  build experiment    : {PIPELINE_EXPERIMENT}")
print(f"  load experiment     : {LOAD_EXPERIMENT}")
print(f"  run snRNA build     : {RUN_BUILD_SNRNA}")
print(f"  run spatial build   : {RUN_BUILD_SPATIAL}")
print(f"  run HLCA mapping    : {RUN_HLCA_MAPPING}")
print(f"  run HLCA eval       : {RUN_HLCA_EVAL}")
print(f"  run Tangram mapping : {RUN_TANGRAM_MAPPING}")


Controls configured.
  data config         : local
  build experiment    : smoke
  load experiment     : smoke
  run snRNA build     : False
  run spatial build   : False
  run HLCA mapping    : False
  run HLCA eval       : False
  run Tangram mapping : True


In [3]:
NB_RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PIPELINE_RUNS: dict[str, str] = {}
HLCA_SUMMARY: dict | None = None
TANGRAM_SUMMARY: dict | None = None


def _run_stagebridge_cli(script_path: str, overrides: list[str], run_id_prefix: str):
    run_id = f"{run_id_prefix}_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        script_path,
        f"data={PIPELINE_DATA}",
        *overrides,
        f"+run_id={run_id}",
    ]
    print("$ " + " ".join(shlex.quote(x) for x in cmd))

    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines: list[str] = []
    assert proc.stdout is not None
    with tqdm(desc=f"{Path(script_path).name} output", unit="line", dynamic_ncols=True) as pbar:
        for line in proc.stdout:
            print(line, end="")
            lines.append(line.rstrip("\n"))
            pbar.update(1)

    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"{script_path} failed with exit code {rc}")
    return run_id, lines


def _parse_last_json_line(lines: list[str]) -> dict | None:
    for raw in reversed(lines):
        text = raw.strip()
        if text.startswith("{") and text.endswith("}"):
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                continue
    return None


if RUN_BUILD_SNRNA:
    run_id, _ = _run_stagebridge_cli(
        "scripts/run_snrna_pipeline.py",
        [f"experiment={PIPELINE_EXPERIMENT}", *PIPELINE_OVERRIDES],
        "nb_snrna",
    )
    PIPELINE_RUNS["snrna"] = run_id

if RUN_BUILD_SPATIAL:
    run_id, _ = _run_stagebridge_cli(
        "scripts/run_spatial_pipeline.py",
        [f"experiment={PIPELINE_EXPERIMENT}", *PIPELINE_OVERRIDES],
        "nb_spatial",
    )
    PIPELINE_RUNS["spatial"] = run_id

if RUN_HLCA_MAPPING:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_hlca_mapping.py",
        [f"experiment={HLCA_EXPERIMENT}", *HLCA_OVERRIDES],
        "nb_hlca",
    )
    PIPELINE_RUNS["hlca"] = run_id
    HLCA_SUMMARY = _parse_last_json_line(lines)
    print("HLCA summary:", HLCA_SUMMARY)

if RUN_HLCA_EVAL:
    run_id, lines = _run_stagebridge_cli(
        "scripts/eval_hlca_mapping.py",
        [*HLCA_EVAL_OVERRIDES],
        "nb_hlca_eval",
    )
    PIPELINE_RUNS["hlca_eval"] = run_id
    eval_summary = _parse_last_json_line(lines)
    print("HLCA eval summary:", eval_summary)

if RUN_TANGRAM_MAPPING:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_tangram_mapping.py",
        [f"experiment={TANGRAM_EXPERIMENT}", *TANGRAM_OVERRIDES],
        "nb_tangram",
    )
    PIPELINE_RUNS["tangram"] = run_id
    TANGRAM_SUMMARY = _parse_last_json_line(lines)
    print("Tangram summary:", TANGRAM_SUMMARY)

if not PIPELINE_RUNS:
    print("No pipeline scripts executed in this run.")
else:
    print("Executed run_ids:", PIPELINE_RUNS)


$ /home/ajbook/micromamba/envs/stagebridge/bin/python scripts/run_tangram_mapping.py data=local experiment=full tangram.show_progress=true tangram.device=cpu tangram.max_training_genes=1000 tangram.num_epochs=200 +run_id=nb_tangram_20260302T154743Z


run_tangram_mapping.py output: 0line [00:00, ?line/s]


Tangram: aggregate HLCA profiles:   0%|          | 0/40 [00:00<?, ?chunk/s]


Tangram: aggregate HLCA profiles:  10%|█         | 4/40 [00:00<00:02, 14.36chunk/s]


Tangram: aggregate HLCA profiles:  20%|██        | 8/40 [00:00<00:02, 14.40chunk/s]


...

[2026-03-02 10:53:31,474][root][INFO] - Printing scores every 100 epochs.


Score: 0.271, KL reg: 0.592
Score: 0.515, KL reg: 0.001
[2026-03-02 11:17:23,417][root][INFO] - Saving results..


[2026-03-02 11:17:46,708][root][INFO] - spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


n_spots=639816
n_training_genes=1000
n_label_profiles_used=9
peak_rss_mb=30637.2890625
{"ok": true, "run_id": "nb_tangram_20260302T154743Z", "mapping_h5ad": "/mnt/e/StageBridge_data/processed/tangram/tangram_map_full.h5ad", "spatial_h5ad": "/mnt/e/StageBridge_data/processed/tangram/spatial_tangram_full.h5ad", "scores_parquet": "/mnt/e/StageBridge_data/processed/tangram/spatial_tangram_celltype_scores.parquet"}


Tangram summary: {'ok': True, 'run_id': 'nb_tangram_20260302T154743Z', 'mapping_h5ad': '/mnt/e/StageBridge_data/processed/tangram/tangram_map_full.h5ad', 'spatial_h5ad': '/mnt/e/StageBridge_data/processed/tangram/spatial_tangram_full.h5ad', 'scores_parquet': '/mnt/e/StageBridge_data/processed/tangram/spatial_tangram_celltype_scores.parquet'}
Executed run_ids: {'tangram': 'nb_tangram_20260302T154743Z'}
